RETRIEVER

In [1]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader,get_response_synthesizer
from llama_index.core import SimpleDirectoryReader
from llama_index.core import Settings

Settings.chunk_size = 128
Settings.chunk_overlap = 50

documents = SimpleDirectoryReader("/mnt/disk1/sjw/llama_index/data").load_data()
index = VectorStoreIndex.from_documents(documents)
retriever = index.as_retriever(verbose=True, similarity_top_k=3)

In [2]:
eval_q_data = SimpleDirectoryReader("/mnt/disk1/sjw/llama_index/eval-q").load_data()

In [3]:
q_data = eval_q_data[0].text

questions = [line.strip().strip('"') for line in q_data.split('\n') if line.strip()]

In [4]:
retriever_result = {}
search_results = []

for count, question in enumerate(questions, start=1):
    search_this = f"{question}"
    search_result = retriever.retrieve(search_this)
    search_results = []
    
    for i in range(retriever.similarity_top_k):
        search_results.append(search_result[i].get_text())

    retriever_result[count] = search_results

EMBEDDING MODEL

VECTOR STORE

In [5]:
from llama_index.core import VectorStoreIndex

vector_index = VectorStoreIndex.from_documents(documents)

GENERATOR

In [6]:
responses=[]

In [7]:
from llama_index.llms.openai import OpenAI
from openai import ChatCompletion
from llama_index.core.llms import ChatMessage

llm = OpenAI(model="gpt-4o-mini")

In [ ]:
for count, question in enumerate(questions, start=1):
    retriever_context_list = retriever_result.get(count, [])
    retriever_context = ' '.join(retriever_context_list)
    
    query = f"Given the context: '{retriever_context}', determine if the following statement is 'True' or 'False': {question}. Start your response with 'True' or 'False'. Answer:"
    messages = [
    ChatMessage(
        role="system", content="You are a helpful assisant"
    ),
    ChatMessage(role="user", content=query),
    ]
    response = llm.chat(messages)

    # Find the position of "Answer:" and extract the response
    #aa = response_text.find("Answer:")
    #if aa != -1:
     #   response = response_text[aa + 7:].strip().split()[0]  # Extract the first word after "Answer:"

    responses.append(response)

    print(count, response)

1 assistant: True
2 assistant: False.
3 assistant: True
4 assistant: False
5 assistant: True
6 assistant: True
7 assistant: True
8 assistant: True
9 assistant: False
10 assistant: False
11 assistant: False.
12 assistant: True
13 assistant: False
14 assistant: True
15 assistant: False.
16 assistant: False
17 assistant: False.
18 assistant: True
19 assistant: False.
20 assistant: True
21 assistant: False.
22 assistant: True
23 assistant: False
24 assistant: False.
25 assistant: True
26 assistant: False.
27 assistant: True
28 assistant: False.
29 assistant: True
30 assistant: True
31 assistant: False.
32 assistant: True
33 assistant: True
34 assistant: False.
35 assistant: True
36 assistant: True
37 assistant: False.
38 assistant: False
39 assistant: True
40 assistant: False
41 assistant: True
42 assistant: False
43 assistant: False.
44 assistant: True
45 assistant: True
46 assistant: False.
47 assistant: True
48 assistant: False
49 assistant: True
50 assistant: False
51 assistant: True
5

EVALUATOR

In [ ]:
eval_a_data = SimpleDirectoryReader("/mnt/disk1/sjw/llama_index/eval-a").load_data()

In [ ]:
a_data = eval_a_data[0].text

answers = [line.strip().strip('"') for line in a_data.split('\n') if line.strip()]

In [ ]:
responses_str = []

In [ ]:
for response in responses:
    response_str = str(response)
    if "True" in response_str:
        response_str = "True"
    elif "False" in response_str:
        response_str = "False"
    
    responses_str.append(response_str)

In [ ]:
correct_count=0
number=0

for answer, response, response_str, question in zip(answers, responses, responses_str, questions,):
    number+=1

    if answer == response_str:
        correct_count += 1
    else: 
        print("<<wrong>>\n", number, question)
        print("RESPONSE", response)
        print("CORRECT ANSWER", answer)
        print()
print(f"correct_count: {correct_count}")

In [ ]:
accuracy = (correct_count / len(questions)) * 100
print(f"Total Questions: {len(questions)}")
print(f"Correct Answers: {correct_count}")
print(f"Accuracy: {accuracy:.2f}%")

In [ ]:
Settings